# Conformer with an Autoregressive Decoder

The fourth architecture, and the one that produced the most useful finding in the project.

A Conformer encoder, identical in spirit to the one in `01-bilstm-and-conformer-ctc.ipynb`, paired with a Transformer decoder that generates gloss tokens one at a time conditioned on its own previous outputs. Trained with cross-entropy and label smoothing rather than CTC.

**It reaches 8.76% dev WER, the lowest number in the project, and it was deliberately not deployed.**

The error analysis at the end explains why, and the short version is that the model had stopped translating: it produces fluent Arabic that no longer depends on the video, and the metric only looked good because predictions were truncated at the first end-of-sequence token before scoring.

The model that shipped is the Conformer with CTC at 13.04%.

Authors: Mohammed Al Sheqaih, Abdulrahman Ammar, Naif Alenazi
Supervisor: Dr. Hamzah Luqman

## 1. Data Acquisition (Download & Unzip)

Downloading the Isharah dataset from Google Drive and unzipping it into the local environment for faster access.

In [2]:
# 1. Install necessary libraries
!pip install -q gdown jiwer

import os
import gdown
import zipfile

# 2. Download Dataset
zip_url = "https://drive.google.com/uc?id=1d9BcEjNkyLsSdE_tzyPiUTuSFbbpuQ8M"
zip_path = "public_si_dat.zip"
data_dir = "public_si_data"

if not os.path.exists(zip_path):
    print("Downloading dataset...")
    gdown.download(zip_url, zip_path, quiet=False)

if not os.path.exists(data_dir):
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(data_dir)
    print("Dataset extracted.")
else:
    print("Dataset already ready.")

# Verify files
print("Files:", os.listdir(data_dir))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.6 MB/s eta 0:00:00


Downloading...
From (original): https://drive.google.com/uc?id=1d9BcEjNkyLsSdE_tzyPiUTuSFbbpuQ8M
From (redirected): https://drive.google.com/uc?id=1d9BcEjNkyLsSdE_tzyPiUTuSFbbpuQ8M&confirm=t&uuid=6fcf3aec-1482-4740-9389-919f3c288136
To: /content/public_si_dat.zip
100%|██████████| 1.68G/1.68G [00:24<00:00, 68.7MB/s]


Extracting dataset...
Dataset extracted.
Files: ['pose_data_isharah1000_hands_lips_body_May12.pkl', 'dev.csv', '__MACOSX', 'train.csv']


In [3]:
import os

# Paths
pkl_path = './isharah_data/pose_data_isharah1000_hands_lips_body_May12.pkl'
zip_path = 'train_data.zip'

# 1. Delete Corrupt File
if os.path.exists(pkl_path):
    print(f" Deleting corrupt file: {pkl_path}")
    os.remove(pkl_path)

# 2. Re-Download the Training Zip (Contains the Pickle)
print(" Re-downloading Training Data (Large File)...")
# Using the ID you provided earlier for training data
!gdown 1d9BcEjNkyLsSdE_tzyPiUTuSFbbpuQ8M -O train_data.zip

# 3. Unzip
print(" Extracting... (This may take a minute)")
!unzip -o -q train_data.zip -d ./isharah_data/

# 4. Verification
if os.path.exists(pkl_path):
    size_gb = os.path.getsize(pkl_path) / (1024**3)
    print(f" Success! File restored. Size: {size_gb:.2f} GB")
    if size_gb < 2.0:
        print("⚠️ WARNING: File looks too small. It should be around 3GB.")
else:
    print(" Error: Extraction failed.")

 Re-downloading Training Data (Large File)...
Downloading...
From (original): https://drive.google.com/uc?id=1d9BcEjNkyLsSdE_tzyPiUTuSFbbpuQ8M
From (redirected): https://drive.google.com/uc?id=1d9BcEjNkyLsSdE_tzyPiUTuSFbbpuQ8M&confirm=t&uuid=298855e9-2a0a-44fe-b275-418304323be6
To: /content/train_data.zip
100% 1.68G/1.68G [00:23<00:00, 71.3MB/s]
 Extracting... (This may take a minute)
 Success! File restored. Size: 3.00 GB


## 2. Data Inspection

Here we inspect the directory structure and verify that the .pkl and .csv files are loaded correctly. This step ensures the data integrity before processing.

In [4]:
import os
import pandas as pd
import pickle
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter

# --- FIX: Point to the folder where you unzipped the data earlier ---
data_dir = "./isharah_data/"

# 1. Load DataFrames
print(f" Loading CSVs from: {data_dir}")
train_df = pd.read_csv(os.path.join(data_dir, "train.csv"))
dev_df = pd.read_csv(os.path.join(data_dir, "dev.csv"))

# 2. Load Pose Data
pkl_path = os.path.join(data_dir, "pose_data_isharah1000_hands_lips_body_May12.pkl")
print(f" Loading Pickle: {pkl_path}")

with open(pkl_path, "rb") as f:
    pose_data = pickle.load(f)

# 3. Filter IDs (Keep only rows that have matching Pose Data)
valid_ids = set(pose_data.keys())
print(f"Original Train size: {len(train_df)}")
train_df = train_df[train_df["id"].isin(valid_ids)].reset_index(drop=True)
dev_df = dev_df[dev_df["id"].isin(valid_ids)].reset_index(drop=True)
print(f"Filtered Train size: {len(train_df)}")

# --- Preprocessing Functions ---
POSE_ARRAY_KEY = "keypoints"
BODY_INDICES = list(range(0, 11))

def get_pose_array(seq_id):
    # Fetch data and ensure it's float32
    return np.asarray(pose_data[str(seq_id)][POSE_ARRAY_KEY], dtype=np.float32)

def augment_pose_sequence(arr):
    """Applies random rotation, scaling, and speed variation."""
    T, J, C = arr.shape

    # 1. Random Speed (Temporal Interpolation)
    if np.random.rand() < 0.5:
        speed = np.random.uniform(0.8, 1.2)
        new_T = max(int(T * speed), 10)
        # Interpolate each coordinate
        arr = np.array([
            [np.interp(np.linspace(0, T-1, new_T), np.arange(T), arr[:, j, c])
             for c in range(C)] for j in range(J)
        ]).transpose(2, 0, 1)
        T = new_T # Update T

    # 2. Random Rotation
    if np.random.rand() < 0.5:
        angle = np.random.uniform(-15, 15) * np.pi / 180
        c, s = np.cos(angle), np.sin(angle)
        rot_mat = np.array([[c, -s], [s, c]])
        # Reshape to (Points, 2) -> Rotate -> Reshape back
        arr_flat = arr.reshape(-1, 2)
        arr_flat = np.dot(arr_flat, rot_mat)
        arr = arr_flat.reshape(T, J, C)

    # 3. Random Scale
    if np.random.rand() < 0.5:
        scale = np.random.uniform(0.9, 1.1)
        arr = arr * scale

    return arr.astype(np.float32)

def pose_to_features(arr):
    """Normalizes and extracts features (coordinates + velocity)."""
    # 1. Center around body mean (Indices 0-11 are body)
    center = arr[:, BODY_INDICES, :].mean(axis=1, keepdims=True)
    arr = arr - center

    # 2. Scale normalization (Standard Deviation)
    scale = arr.std() + 1e-6
    arr = arr / scale

    # 3. Flatten (Time, Joints*Coords)
    T, J, C = arr.shape
    feat = arr.reshape(T, -1)

    # 4. Add Velocity (Motion)
    velocity = np.zeros_like(feat)
    velocity[1:] = feat[1:] - feat[:-1]

    return np.concatenate([feat, velocity], axis=-1)

print(" Data Loading & Preprocessing Ready.")

 Loading CSVs from: ./isharah_data/
 Loading Pickle: ./isharah_data/pose_data_isharah1000_hands_lips_body_May12.pkl


/tmp/ipython-input-3966707986.py:23: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  pose_data = pickle.load(f)


Original Train size: 10000
Filtered Train size: 9500
 Data Loading & Preprocessing Ready.


## 3. Vocabulary Builder & Dataset & DataLoader Definition

We define a `Vocabulary` class mapping gloss words to indices, the `IsharahDataset` that loads and normalises pose keypoints, and the `DataLoader` that batches them.

Note that this notebook builds its **own vocabulary**, with `<sos>` and `<eos>` tokens the CTC models do not need, so its size differs from theirs. The two are not interchangeable.

In [5]:
# Build Vocabulary
all_glosses = list(train_df["gloss"]) + list(dev_df["gloss"])
counter = Counter()
for g in all_glosses:
    counter.update(str(g).split())

# Special tokens for Seq2Seq
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
SOS_TOKEN = "<sos>" # Start of Sentence
EOS_TOKEN = "<eos>" # End of Sentence

vocab = [PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN] + [w for w, _ in counter.most_common()]
token2id = {w: i for i, w in enumerate(vocab)}
id2token = {i: w for i, w in enumerate(vocab)}
VOCAB_SIZE = len(vocab)

print(f"Vocabulary Size: {VOCAB_SIZE}")

# Dataset Class
class IsharahDataset(Dataset):
    def __init__(self, df, training=False):
        self.df = df
        self.training = training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pose = get_pose_array(row["id"])

        # Augment only if training
        if self.training:
            pose = augment_pose_sequence(pose)

        features = pose_to_features(pose)

        # Tokenize targets
        tokens = str(row["gloss"]).split()
        # Add <sos> and <eos> for the LLM decoder
        target_ids = [token2id[SOS_TOKEN]] + \
                     [token2id.get(t, token2id[UNK_TOKEN]) for t in tokens] + \
                     [token2id[EOS_TOKEN]]

        return {
            "pose": torch.tensor(features, dtype=torch.float32),
            "targets": torch.tensor(target_ids, dtype=torch.long)
        }

def collate_fn(batch):
    poses = [x["pose"] for x in batch]
    targets = [x["targets"] for x in batch]
    pose_lens = torch.tensor([len(p) for p in poses])
    target_lens = torch.tensor([len(t) for t in targets])

    # Pad sequences
    poses_pad = pad_sequence(poses, batch_first=True, padding_value=0.0)
    targets_pad = pad_sequence(targets, batch_first=True, padding_value=token2id[PAD_TOKEN])

    return poses_pad, targets_pad, pose_lens, target_lens

# DataLoaders
train_loader = DataLoader(IsharahDataset(train_df, training=True), batch_size=16, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(IsharahDataset(dev_df, training=False), batch_size=16, shuffle=False, collate_fn=collate_fn)

Vocabulary Size: 679


## 4. The Model

### The encoder: ConformerBlock

A Macaron-style block, sandwiching self-attention and convolution between two feed-forward networks. Attention carries global sequence context and the convolution captures fine-grained local movement.

### The full network: sequence to sequence

Four `ConformerBlock`s encode the raw pose into deep features, and a standard Transformer decoder attends over those encodings to generate the gloss sequence autoregressively.

**This is the one architectural difference that matters in this project.** The other three models predict each output position independently under CTC, which constrains the output to align monotonically with the input. This one generates each token conditioned on the tokens it has already produced, and has no such constraint. The consequences are the subject of the error analysis at the end.

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- 1. Fixed Conformer Block (Corrects the Shape Error) ---
class ConformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, conv_kernel, dropout=0.1):
        super().__init__()
        # FFN 1
        self.ffn1 = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model*4), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(d_model*4, d_model), nn.Dropout(dropout)
        )

        # Attention
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)

        # Convolution Module
        self.norm2 = nn.LayerNorm(d_model) # Normalization BEFORE Transpose
        self.conv = nn.Sequential(
            nn.Conv1d(d_model, d_model*2, 1), nn.GLU(dim=1),
            nn.Conv1d(d_model, d_model, conv_kernel, groups=d_model, padding=conv_kernel//2),
            nn.BatchNorm1d(d_model), nn.SiLU(),
            nn.Conv1d(d_model, d_model, 1), nn.Dropout(dropout)
        )

        # FFN 2
        self.ffn2 = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model*4), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(d_model*4, d_model), nn.Dropout(dropout)
        )
        self.norm_final = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        # 1. FFN 1 (Macaron)
        x = x + 0.5 * self.ffn1(x)

        # 2. Attention
        # Note: key_padding_mask=mask (True means Padding, so we ignore it)
        attn_out, _ = self.attn(self.norm1(x), self.norm1(x), self.norm1(x),
                                key_padding_mask=mask)
        x = x + attn_out

        # 3. Convolution
        residual = x
        x_norm = self.norm2(x)             # [B, T, D] -> Normalize last dim
        x_in = x_norm.transpose(1, 2)      # [B, D, T] -> Swap for Conv1d
        x_out = self.conv(x_in)            # [B, D, T] -> Conv
        x = residual + x_out.transpose(1, 2) # [B, T, D] -> Swap back

        # 4. FFN 2
        x = x + 0.5 * self.ffn2(x)

        return self.norm_final(x)

# --- 2. Re-Define ConformerLLM (To use the fixed Block) ---
class ConformerLLM(nn.Module):
    def __init__(self, input_dim, vocab_size, d_model=256):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc = nn.Parameter(torch.randn(1, 1000, d_model) * 0.02)

        self.encoder_layers = nn.ModuleList([
            ConformerBlock(d_model, n_heads=4, conv_kernel=31) for _ in range(4)
        ])

        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=4, dim_feedforward=1024, batch_first=True)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=3)
        self.tgt_emb = nn.Embedding(vocab_size, d_model)
        self.tgt_pos = nn.Parameter(torch.randn(1, 100, d_model) * 0.02)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def encode(self, x, mask):
        x = self.input_proj(x) + self.pos_enc[:, :x.size(1), :]
        for layer in self.encoder_layers:
            x = layer(x, mask)
        return x

    def forward(self, src, tgt, src_mask=None, tgt_mask=None, tgt_key_padding_mask=None):
        memory = self.encode(src, src_mask)
        tgt_emb = self.tgt_emb(tgt) + self.tgt_pos[:, :tgt.size(1), :]
        output = self.decoder(tgt_emb, memory, tgt_mask=tgt_mask,
                              memory_key_padding_mask=src_mask, # Encoder mask
                              tgt_key_padding_mask=tgt_key_padding_mask) # Decoder mask
        return self.fc_out(output)

## 6. Training Configuration & Utilities

Hyperparameters, the AdamW optimiser, and the loss.

**The loss here is cross-entropy with label smoothing of 0.1, not CTC.** An autoregressive decoder predicts one token at a time against a shifted target, so the alignment-free CTC objective used by the other three models does not apply. Word error rate is still the evaluation metric, computed with `jiwer` after decoding.

Note the decoding path, because it matters for reading the numbers below. `evaluate()` truncates each prediction at its **first `<eos>` token** before scoring, so anything generated after that point never reaches the metric.

In [ ]:
import jiwer
import torch
import torch.nn as nn
import os
from google.colab import drive

drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/Isharah_Project_Models'
os.makedirs(save_dir, exist_ok=True)

best_model_path = os.path.join(save_dir, 'best_conformer_llm.pth')

print(f" Models will be saved PERMANENTLY to: {save_dir}")

if 'model' not in globals():
    sample_batch = next(iter(train_loader))
    input_dim = sample_batch[0].shape[-1]
    VOCAB_SIZE = len(token2id)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = ConformerLLM(input_dim=input_dim, vocab_size=VOCAB_SIZE).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(ignore_index=token2id[PAD_TOKEN], label_smoothing=0.1)

def generate_square_subsequent_mask(sz):
    mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask.to(device)

def train_epoch(model, loader):
    model.train()
    total_loss = 0
    for poses, targets, pose_lens, target_lens in loader:
        poses, targets = poses.to(device), targets.to(device)
        dec_input = targets[:, :-1]
        labels = targets[:, 1:]
        B, T = poses.shape[:2]
        L = dec_input.shape[1]
        src_mask = torch.arange(T, device=device)[None, :] >= pose_lens[:, None].to(device)
        tgt_pad_mask = (dec_input == token2id[PAD_TOKEN])
        tgt_causal_mask = generate_square_subsequent_mask(L)

        optimizer.zero_grad()
        logits = model(poses, dec_input,
                       src_mask=src_mask,
                       tgt_key_padding_mask=tgt_pad_mask,
                       tgt_mask=tgt_causal_mask)
        loss = criterion(logits.reshape(-1, VOCAB_SIZE), labels.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def decode_batch(model, poses, pose_lens):
    model.eval()
    B = poses.size(0)
    poses = poses.to(device)
    src_mask = torch.arange(poses.size(1), device=device)[None, :] >= pose_lens[:, None].to(device)
    with torch.no_grad():
        memory = model.encode(poses, src_mask)
        curr_tokens = torch.full((B, 1), token2id[SOS_TOKEN], dtype=torch.long, device=device)
        for _ in range(50):
            tgt_mask = generate_square_subsequent_mask(curr_tokens.size(1))
            out = model.decoder(model.tgt_emb(curr_tokens) + model.tgt_pos[:, :curr_tokens.size(1)],
                                memory, tgt_mask=tgt_mask, memory_key_padding_mask=src_mask)
            next_token = model.fc_out(out[:, -1, :]).argmax(dim=-1, keepdim=True)
            curr_tokens = torch.cat([curr_tokens, next_token], dim=1)
            if (next_token == token2id[EOS_TOKEN]).all(): break
    return curr_tokens.cpu().numpy()

def evaluate(model, loader):
    preds, refs = [], []
    for poses, targets, p_lens, t_lens in loader:
        generated = decode_batch(model, poses, p_lens)
        for i in range(len(poses)):
            p_sent = []
            for idx in generated[i][1:]:
                if idx == token2id[EOS_TOKEN]: break
                p_sent.append(id2token.get(idx, ""))
            r_sent = []
            for idx in targets[i][1:]:
                idx = idx.item()
                if idx == token2id[EOS_TOKEN]: break
                if idx != token2id[PAD_TOKEN]: r_sent.append(id2token.get(idx, ""))
            preds.append(" ".join(p_sent))
            refs.append(" ".join(r_sent))
    return jiwer.wer(refs, preds), preds[0], refs[0]

print(" Starting Training (Saving to Google Drive)...")
best_wer = float('inf')

for epoch in range(30):
    loss = train_epoch(model, train_loader)
    wer, sample_pred, sample_ref = evaluate(model, dev_loader)

    print(f"Epoch {epoch+1:02d} | Loss: {loss:.4f} | Dev WER: {wer:.4f}")

    # --- SAVE LOGIC (To Drive) ---
    if wer < best_wer:
        best_wer = wer
        torch.save(model.state_dict(), best_model_path)
        print(f"    Saved to Drive! (WER: {best_wer:.4f}) at {best_model_path}")

    if epoch % 5 == 0:
        print(f"   Ref:  {sample_ref}")
        print(f"   Pred: {sample_pred}")

Mounted at /content/drive
 Models will be saved PERMANENTLY to: /content/drive/MyDrive/Isharah_Project_Models
 Starting Training (Saving to Google Drive)...


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Epoch 01 | Loss: 4.0419 | Dev WER: 0.9708
    Saved to Drive! (WER: 0.9708) at /content/drive/MyDrive/Isharah_Project_Models/best_conformer_llm.pth
   Ref:  سوال هو
   Pred: انا حب اكل
Epoch 02 | Loss: 2.5722 | Dev WER: 1.1662
Epoch 03 | Loss: 2.0454 | Dev WER: 0.9585
    Saved to Drive! (WER: 0.9585) at /content/drive/MyDrive/Isharah_Project_Models/best_conformer_llm.pth
Epoch 04 | Loss: 1.7734 | Dev WER: 0.6860
    Saved to Drive! (WER: 0.6860) at /content/drive/MyDrive/Isharah_Project_Models/best_conformer_llm.pth
Epoch 05 | Loss: 1.5672 | Dev WER: 0.5463
    Saved to Drive! (WER: 0.5463) at /content/drive/MyDrive/Isharah_Project_Models/best_conformer_llm.pth
Epoch 06 | Loss: 1.4201 | Dev WER: 0.3377
    Saved to Drive! (WER: 0.3377) at /content/drive/MyDrive/Isharah_Project_Models/best_conformer_llm.pth
   Ref:  سوال هو
   Pred: سوال كوب شرب
Epoch 07 | Loss: 1.3294 | Dev WER: 0.2879
    Saved to Drive! (WER: 0.2879) at /content/drive/MyDrive/Isharah_Project_Models/best_conformer_ll

In [17]:
def decode_batch(model, poses, pose_lens, repetition_penalty=1.2):
    model.eval()
    B = poses.size(0)
    poses = poses.to(device)

    # Create mask for the encoder
    src_mask = torch.arange(poses.size(1), device=device)[None, :] >= pose_lens[:, None].to(device)

    with torch.no_grad():
        memory = model.encode(poses, src_mask)
        curr_tokens = torch.full((B, 1), token2id[SOS_TOKEN], dtype=torch.long, device=device)

        # We need to track which sentences in the batch are finished
        finished = torch.zeros(B, dtype=torch.bool, device=device)

        for _ in range(50):
            tgt_mask = generate_square_subsequent_mask(curr_tokens.size(1))

            out = model.decoder(
                model.tgt_emb(curr_tokens) + model.tgt_pos[:, :curr_tokens.size(1)],
                memory,
                tgt_mask=tgt_mask,
                memory_key_padding_mask=src_mask
            )

            # Get the logits for the last token
            logits = model.fc_out(out[:, -1, :]) # Shape: [Batch, Vocab_Size]

            # --- APPLY REPETITION PENALTY ---
            # This loops through every sample in the batch
            for i in range(B):
                # Look at tokens already generated for this sentence
                for previous_token_id in curr_tokens[i]:
                    # If the logit is positive, divide by penalty. If negative, multiply.
                    # This lowers the score of words we already said.
                    if logits[i, previous_token_id] < 0:
                        logits[i, previous_token_id] *= repetition_penalty
                    else:
                        logits[i, previous_token_id] /= repetition_penalty

            # Select the best token (Greedy)
            next_token = logits.argmax(dim=-1, keepdim=True)

            # Logic to stop updating sentences that hit EOS
            # If a sentence is already finished, we just append PAD (or keep EOS)
            # This prevents the "Infinite Loop" from messing up the metric
            next_token = torch.where(
                finished.unsqueeze(1),
                torch.tensor(token2id[PAD_TOKEN], device=device),
                next_token
            )

            curr_tokens = torch.cat([curr_tokens, next_token], dim=1)

            # Update finished status
            current_eos = (next_token == token2id[EOS_TOKEN]).squeeze(1)
            finished = finished | current_eos

            # If all sentences are finished, stop immediately
            if finished.all():
                break

    return curr_tokens.cpu().numpy()

## 7. Error Analysis

In [22]:
import torch
import pandas as pd
import numpy as np
from collections import Counter

def generate_detailed_report(model, loader, device, max_samples=200):
    """
    Generates a comprehensive error analysis report including:
    1. Top Missed Words
    2. Model Bias (Length Analysis)
    3. Categorized Examples (Deletion vs Substitution)
    """
    model.eval()

    # Storage for stats
    missed_words_counter = Counter()
    length_diffs = []

    deletion_errors = []
    substitution_errors = []
    insertion_errors = []

    print(f"Analyzing {max_samples} samples... please wait.")

    processed_count = 0

    with torch.no_grad():
        for poses, targets, p_lens, t_lens in loader:
            # Use the Aggressive Beam Search (Best for preventing loops)
            generated_seqs = decode_aggressive(model, poses, p_lens, beam_width=5, alpha=1.5)

            for i in range(len(poses)):
                if processed_count >= max_samples:
                    break

                # --- 1. Decode Strings ---
                # Prediction
                pred_ids = generated_seqs[i].squeeze().cpu().numpy()
                pred_tokens = []
                for idx in pred_ids:
                    if idx == token2id[SOS_TOKEN]: continue
                    if idx == token2id[EOS_TOKEN]: break
                    pred_tokens.append(id2token.get(idx, ""))
                pred_text = " ".join(pred_tokens).strip()

                # Ground Truth
                ref_tokens = []
                for idx in targets[i]:
                    idx = idx.item()
                    if idx == token2id[SOS_TOKEN]: continue
                    if idx == token2id[EOS_TOKEN]: break
                    if idx != token2id[PAD_TOKEN]:
                        ref_tokens.append(id2token.get(idx, ""))
                ref_text = " ".join(ref_tokens).strip()

                # --- 2. Calculate Stats ---
                # Length Difference
                len_diff = len(pred_tokens) - len(ref_tokens)
                length_diffs.append(len_diff)

                # Missed Words (Set difference)
                ref_set = set(ref_tokens)
                pred_set = set(pred_tokens)
                missing = list(ref_set - pred_set)
                missed_words_counter.update(missing)

                missing_str = ", ".join(missing) if missing else "-"

                # --- 3. Categorize Errors ---
                error_entry = {
                    "Reference": ref_text,
                    "Prediction": pred_text,
                    "Missing Words": missing_str,
                    "Length Diff": len_diff
                }

                if pred_text != ref_text:
                    if len(pred_tokens) < len(ref_tokens):
                        deletion_errors.append(error_entry)
                    elif len(pred_tokens) > len(ref_tokens):
                        insertion_errors.append(error_entry)
                    else:
                        substitution_errors.append(error_entry)

                processed_count += 1

            if processed_count >= max_samples:
                break

    # --- 4. PRINT REPORT ---
    print("\n" + "="*40)
    print("--- DETAILED ERROR ANALYSIS ---")
    print("="*40 + "\n")

    # A. Top Missed Words
    print("Top 10 Most Missed Words (Hardest signs for the model):")
    for word, count in missed_words_counter.most_common(10):
        print(f"   - '{word}': Missed {count} times")

    # B. Bias Analysis
    avg_diff = np.mean(length_diffs)
    bias_desc = "Neutral"
    if avg_diff < -0.5: bias_desc = "High Deletion Rate (Predicts too few words)"
    elif avg_diff > 0.5: bias_desc = "High Insertion Rate (Predicts too many words)"

    print(f"\nModel Bias Analysis (Average Length Difference: {avg_diff:.2f}):")
    print(f"   Result: {bias_desc}")

    # C. Dataframes
    pd.set_option('display.max_colwidth', None)

    print("\n--- Examples: Deletion Errors (Prediction is too short) ---")
    if deletion_errors:
        df_del = pd.DataFrame(deletion_errors).drop(columns=['Length Diff']).head(5)
        print(df_del.to_markdown(index=False))
    else:
        print("No deletion errors found.")

    print("\n--- Examples: Substitution Errors (Word Confusion) ---")
    if substitution_errors:
        df_sub = pd.DataFrame(substitution_errors).drop(columns=['Length Diff']).head(5)
        print(df_sub.to_markdown(index=False))
    else:
        print("No substitution errors found.")

    print("\n--- Examples: Insertion Errors (Prediction is too long) ---")
    if insertion_errors:
        df_ins = pd.DataFrame(insertion_errors).drop(columns=['Length Diff']).head(5)
        print(df_ins.to_markdown(index=False))
    else:
        print("No insertion errors found (Good job controlling loops!).")

# --- Run the Report ---
generate_detailed_report(model, dev_loader, device, max_samples=200)

Analyzing 200 samples... please wait.

--- DETAILED ERROR ANALYSIS ---

Top 10 Most Missed Words (Hardest signs for the model):
   - 'انا': Missed 59 times
   - 'هو': Missed 56 times
   - 'سوال': Missed 31 times
   - 'رغبه': Missed 24 times
   - 'ذهاب': Missed 18 times
   - 'اكل': Missed 16 times
   - 'الان': Missed 15 times
   - 'مدرسه': Missed 13 times
   - 'صديق': Missed 13 times
   - 'وقت': Missed 12 times

Model Bias Analysis (Average Length Difference: 10.86):
   Result: High Insertion Rate (Predicts too many words)

--- Examples: Deletion Errors (Prediction is too short) ---
No deletion errors found.

--- Examples: Substitution Errors (Word Confusion) ---
No substitution errors found.

--- Examples: Insertion Errors (Prediction is too long) ---
| Reference            | Prediction                                                                        | Missing Words            |
|:---------------------|:-----------------------------------------------------------------------------

## Error Analysis Summary

**Best dev WER: 8.76%** at epoch 23, the lowest figure in the project. This model was not the one deployed, and the analysis below is why.

### 1. The model over-generates

**Average length difference: +10.86.** Asked for a three-word sentence, it emits thirteen or fourteen. It fails to produce the `<eos>` token and keeps generating until it hits the decode limit.

### 2. The most-missed words are the most common ones

Top missed: `انا` (I), `هو` (he), `سوال` (question).

These are not hard signs; they are the most frequent words in the corpus. When the output has stopped tracking the input, the words that appear in the most references are the ones missed most often.

### 3. Mode collapse

The model falls into repetition loops, repeating high-frequency tokens such as `اب` or phrases like `الم اسبوع_الماضي` regardless of what the video contains. The decoder has stopped attending to the encoder and is running on its own language prior: fluent-looking Arabic, unconditioned on the signing.

### Reconciling 8.76% with all of the above

A model that babbles eleven extra words per sentence cannot also score 8.76% WER. Both observations are correct; they measure different things.

`evaluate()` truncates each prediction at its first `<eos>` before scoring, so the metric sees a cleaned-up sequence while this analysis sees the raw generation. Two further details compound it:

- **`decode_batch` is defined twice.** The version used during training stops only when *every* sequence in the batch emits `<eos>`, and otherwise runs the full 50 steps. A later cell redefines it with a repetition penalty and per-sequence stopping. This analysis ran after that redefinition, so it did not use the decoder that produced the training log.
- **The training cell carries stored output but no execution count**, so its log predates the notebook's current state.

### Conclusion

This model is not translating. It is producing statistically plausible Arabic that has stopped depending on the pose input, and its low WER is an artefact of scoring a truncated version of that output.

The lesson is worth more than the number: **a lower validation score does not mean a better system.** Reading what a model actually emits catches what a leaderboard cannot. The Conformer with CTC at 13.04% was submitted on the test set; this one was not.